# Practice 35 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: `np.select()`

Nested `np.where` works but gets unreadable fast:

```python
# Hard to read with 3+ conditions
np.where(x > 10000, 'premium', np.where(x > 3000, 'mid', 'budget'))
```

`np.select()` is the cleaner alternative — pass a list of conditions and a matching list of choices:

```python
conditions = [
    df['price'] > 10000,
    df['price'] > 3000,
]
choices = ['premium', 'mid']

df['tier'] = np.select(conditions, choices, default='budget')
```

Conditions are checked **top to bottom** — the first match wins. `default` is used when none match.

---
### Task 1: tips — `np.select()` in practice

Load `sns.load_dataset('tips')`.

1. Use `np.select()` to add a `tip_tier` column: `'generous'` (tip_pct > 20%), `'normal'` (15–20%), `'low'` (< 15%). First add `tip_pct` vectorized.
2. Use `np.select()` to add a `party_label`: `'large party'` (size ≥ 5), `'medium'` (size 3–4), `'couple or solo'` (size ≤ 2).
3. Rewrite task 2 using nested `np.where` instead — confirm the results match. Which is more readable?
4. `groupby('tip_tier')['total_bill'].mean()` — do generous tippers spend more? No `observed=True` needed here since `tip_tier` is not a category column.

In [ ]:
# Your code here

tips = sns.load_dataset('tips')
tips['tip_pct'] = tips['tip']/tips['total_bill']

conditions = [tips['tip_pct']>0.2, tips['tip_pct']>0.15]
choices = ['generous','normal']
tips['tip_tier'] = np.select(conditions, choices, default='low')


condi = [tips['size']>=5, tips['size']>=3]
choi = ['large party', 'medium']

tips['party_label'] = np.select(condi, choi, default='couple or solo')


tips['party_label_w'] = np.where(tips['size']>=5, 'large party', np.where(tips['size']>=3, 'medium','couple or solo'))

pd.crosstab(tips['party_label'], tips['party_label_w'])

### np.select is more readable 

tips.groupby('tip_tier')['total_bill'].mean()

### generous group doesn't spend the most, and is actually the lowest

tip_tier
generous    15.877692
low         22.681376
normal      18.086146
Name: total_bill, dtype: float64

---
## Level 2 — Rewrite and compare

### Task 2: diamonds — five rewrites, then time them

Load `sns.load_dataset('diamonds')`. Rewrite each snippet without `.apply()`. For each one, note **why** the vectorized version is better (speed? readability? both?).

```python
# A
diamonds['depth_per_table'] = diamonds.apply(
    lambda row: row['depth'] / row['table'], axis=1)

# B
diamonds['long_name'] = diamonds['cut'].apply(
    lambda x: True if len(str(x)) > 4 else False)

# C
diamonds['price_bucket'] = diamonds['price'].apply(
    lambda x: 'cheap' if x < 1000
    else 'moderate' if x < 5000
    else 'expensive' if x < 10000
    else 'luxury')

# D
diamonds['sqrt_carat'] = diamonds['carat'].apply(lambda x: x ** 0.5)

# E
diamonds['cut_color'] = diamonds.apply(
    lambda row: str(row['cut']) + '-' + str(row['color']), axis=1)
```

Use `%timeit` to compare A and C (apply vs vectorized).

In [ ]:
diamonds = sns.load_dataset('diamonds')

# Your rewrites here
#A
diamonds['depth_per_table'] = diamonds['depth']/diamonds['table']

#B
diamonds['long_name'] = diamonds['cut'].str.len()>4

#C
conditions = [diamonds['price'] < 1000, diamonds['price']< 5000, diamonds['price']<10000]
choices = ['cheap','modereate','expensive']
diamonds['price_bucket'] = np.select(conditions, choices, default='luxury')

#D
diamonds['sqrt_carat'] = diamonds['carat']** 0.5

#E
diamonds['cut_color'] = diamonds['cut'].astype(str)+"_"+diamonds['color'].astype(str)


%timeit diamonds['depth_per_table'] = diamonds.apply(lambda row: row['depth'] / row['table'], axis=1)
%timeit diamonds['depth_per_table'] = diamonds['depth']/diamonds['table']

%timeit diamonds['price_bucket'] = diamonds['price'].apply(lambda x: 'cheap' if x < 1000 else 'moderate' if x < 5000 else 'expensive' if x < 10000 else 'luxury')

conditions = [diamonds['price'] < 1000, diamonds['price']< 5000, diamonds['price']<10000]
choices = ['cheap','modereate','expensive']
%timeit diamonds['price_bucket'] = np.select(conditions, choices, default='luxury')
### for C np.select is not too fast 

469 ms ± 20 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
207 µs ± 3.99 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
10.4 ms ± 125 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.97 ms ± 209 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


---
## Level 3 — Open questions

### Task 3: titanic — vectorized survival analysis

Load `sns.load_dataset('titanic')`. No `.pipe()`, no `.apply()` anywhere in this task.

Answer these questions using only vectorized operations:

1. Add a `fare_class` column using `np.select()`: `'high'` (fare > 50), `'mid'` (fare 15–50), `'low'` (fare < 15).
2. Add `is_child` (age < 18) and `is_alone` (sibsp + parch == 0) — both vectorized boolean columns.
3. What is the survival rate for children travelling alone vs children not alone? Use `.loc` to filter, `np.mean()` for the rate.
4. Among adults (`is_child == False`), which `fare_class` has the highest survival rate? Use `groupby`, `observed=True` not needed since `fare_class` is object dtype.
5. Add `log_fare` using `np.log`. What is `np.corrcoef` between `log_fare` and `survived`? Is fare correlated with survival?

In [40]:
# Your code here
titanic = sns.load_dataset('titanic')

condi = [titanic['fare']>50, titanic['fare']>15]
choi = ['high','mid']
titanic['fare_class'] = np.select(condi, choi, default='low')

titanic['is_child'] = titanic['age']<18
titanic['is_alone'] = titanic['sibsp']+titanic['parch'] == 0

c = titanic.loc[titanic['is_child']]
c.groupby('is_alone')['survived'].mean()

a = titanic.loc[titanic['is_child'] == False]
a.groupby('fare_class')['survived'].mean()


t = titanic.dropna().copy()
t['log_fare'] = np.log(t['fare']+1)


np.corrcoef(t['log_fare'], t['survived'])

array([[1.        , 0.20077957],
       [0.20077957, 1.        ]])

---
## Level 4 — Pipe

### Task 4: penguins — vectorized pipeline

Load `sns.load_dataset('penguins')`. Drop nulls.

Write a `.pipe()` chain with 3 functions — no `.apply()` allowed anywhere:

- One converts `species` and `island` to `category`, and adds `bill_ratio` (`bill_length_mm / bill_depth_mm`) vectorized
- One uses `transform` to add species-level mean `bill_ratio` and `body_mass_g` — then adds boolean columns `above_avg_bill` and `above_avg_mass` (both vectorized)
- One uses `np.select()` to classify each penguin: `'elite'` (both above average), `'below'` (both below), `'mixed'` otherwise

After the chain:
- Pivot table: count of penguins by `class` × `species`, `observed=True`.
- Which species has the highest fraction of `'elite'` penguins? Use `np.argmax()`.
- Use `.loc` to extract `'elite'` penguins. What is their mean `body_mass_g` vs the full dataset?

In [50]:
# Your code here

penguins = sns.load_dataset('penguins').dropna()

def cov(df):
    df[['species','island']] = df[['species','island']].astype('category')
    df['bill_ratio'] = df['bill_length_mm']/df['bill_depth_mm']
    return(df)
def tras(df):
    df['gbill'] = df.groupby('species', observed = True)['bill_ratio'].transform('mean')
    df['gbm'] = df.groupby('species', observed = True)['body_mass_g'].transform('mean')
    df['above_avg_bill'] = df['bill_ratio']> df['gbill']
    df['above_avg_mass'] = df['body_mass_g']> df['gbm']
    return(df)
def sel(df):
    conditions = [df['above_avg_bill'] & df['above_avg_mass'], 
                  ~df['above_avg_bill'] & ~df['above_avg_mass']]
    choices = ['elite','below']
    df['class'] = np.select(conditions, choices, default='mixed')
    return df 


res = penguins.copy().pipe(cov).pipe(tras).pipe(sel)

p = res.pivot_table(
    index = 'class',
    columns = 'species',
    values = "sex", 
    aggfunc = 'count',
    observed = True
)
p


species,Adelie,Chinstrap,Gentoo
class,,,
below,39,17,28
elite,38,15,24
mixed,69,36,67


In [60]:
n = res[res['class']=='elite'].groupby('species', observed=True)['class'].size()
d = res.groupby('species', observed=True)['class'].size()
f = n/d
f.index[np.argmax(f)]

ee = res.loc[res['class'] == 'elite']
em = ee['body_mass_g'].mean()
rm = res['body_mass_g'].mean()
print(em, rm)


4553.896103896104 4207.057057057057


After the chain:
- Pivot table: count of penguins by `class` × `species`, `observed=True`.
- Which species has the highest fraction of `'elite'` penguins? Use `np.argmax()`.
- Use `.loc` to extract `'elite'` penguins. What is their mean `body_mass_g` vs the full dataset?

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,bill_ratio,gbill,gbm,above_avg_bill,above_avg_mass,class
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,2.090909,2.121478,3706.164384,False,True,mixed
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,2.270115,2.121478,3706.164384,True,True,elite
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,2.238889,2.121478,3706.164384,True,False,mixed
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,1.901554,2.121478,3706.164384,False,False,below
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male,1.907767,2.121478,3706.164384,False,False,below
...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female,3.445255,3.176602,5092.436975,True,False,mixed
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,3.272727,3.176602,5092.436975,True,False,mixed
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,3.210191,3.176602,5092.436975,True,True,elite
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,3.054054,3.176602,5092.436975,False,True,mixed
